# 🧪 Hands-on
### Module 2 · Lecture 4 — *RAG*

**What we'll build:** a small question-answering assistant for a fictional company, **Acme Corp**, that answers employee questions (leave, IT errors, expenses, benefits) **from Acme's own documents**, and cites the source.

We will build every stage of the pipeline from the lecture, one at a time:

| # | Stage (from the slides) | What you'll do here | Main library |
|---|---|---|---|
| 1 | **Document processing** | Read Markdown + a PDF, detect a scanned page, (optional) OCR it | `pypdf`, `pytesseract` |
| 2 | **Chunking** | Fixed-size, token sliding window, recursive, structure-aware, semantic | `transformers`, `langchain-text-splitters` |
| 3 | **Embeddings** | Sentence embeddings, cosine similarity, bi- vs cross-encoder, E5 prefixes | `sentence-transformers` |
| 4 | **Vector store** | FAISS (Flat → IVF → PQ → HNSW) and ChromaDB with metadata filters | `faiss-cpu`, `chromadb` |
| 5 | **Retrieval & reranking** | BM25, dense, hybrid (RRF), cross-encoder reranking, then generate a grounded answer | `rank_bm25`, `transformers` |

…and then we rebuild the same pipeline in a few lines with **LangChain**, and run a mini **bake-off** to measure which retriever is best *on our own data*.

> **How to use this notebook**
> - Run cells top to bottom with **Shift + Enter**.
> - Cells marked **⏩ OPTIONAL** are extras; we may skip them live — try them at home.
> - Every section ends with **🔗 Read more** links to the official documentation and the original papers.
> - Built to run on **Google Colab** (free tier is fine). A GPU is *not* required, but makes the LLM step faster: `Runtime → Change runtime type → T4 GPU`.

### A good RAG system is mostly careful plumbing: clean text, sensible chunks, the right metric, and a quick check on your own questions. The LLM is the last and easiest step.

## Session plan

| Section | Core / optional |
|---|---|
|0 · Setup & installs | core |
|1 · Our knowledge base (Acme Corp) | core |
|2 · Sparse retrieval with BM25 — and where it breaks | core |
|3 · Dense embeddings, bi- vs cross-encoder, E5 prefix trap | core |
|4 · Document processing: PDFs & the scanned-page trap | core (OCR optional) |
|5 · Chunking strategies | core (semantic optional) |
|6 · Vector stores: FAISS & ChromaDB | core (PQ/HNSW optional) |
|7 · Hybrid search with Reciprocal Rank Fusion | core |
|8 · Rerank + generate: the full `ask()` pipeline | core |
|9 · Silent bugs checklist & wrap-up | core |
|10 · LangChain version · 11 · Mini bake-off · Exercises | optional |

---
# 0 · Setup

Colab already ships with `torch`, `transformers` and `sentence-transformers`. We install the rest.
Takes ~1 minute — start it now and read on while it runs.

| Library | What it does | Docs |
|---|---|---|
| `sentence-transformers` | Embedding models (bi-encoders) and cross-encoders | https://sbert.net |
| `transformers` | Tokenizers + the small LLM we use to generate answers | https://huggingface.co/docs/transformers |
| `faiss-cpu` | Fast vector similarity search (Meta AI) | https://github.com/facebookresearch/faiss/wiki |
| `chromadb` | A vector **database**: vectors + text + metadata + filters | https://docs.trychroma.com |
| `rank_bm25` | BM25 keyword (sparse) retrieval | https://github.com/dorianbrown/rank_bm25 |
| `pypdf` | Extract text from PDFs | https://pypdf.readthedocs.io |
| `reportlab` | We use it only to *create* a sample PDF | https://docs.reportlab.com |
| `langchain-*` | Framework that wraps all of the above | https://docs.langchain.com/oss/python/langchain/overview |

In [ ]:
%%capture
# %%capture hides the long pip output. Remove it if something fails, to see the error.
!pip install -q faiss-cpu chromadb rank_bm25 pypdf reportlab
!pip install -q langchain langchain-core langchain-text-splitters langchain-huggingface langchain-chroma

In [ ]:
import os, re, math, time, textwrap, warnings
from pprint import pprint
import numpy as np
import torch

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", DEVICE)

def show(text, width=100):
    """Pretty-print long text with wrapping."""
    print(textwrap.fill(text, width=width))

Running on: cuda


### Models we'll use (all free, all download from the Hugging Face Hub)

| Role | Model | Size | Model card |
|---|---|---|---|
| Bi-encoder (embeddings) | `sentence-transformers/all-MiniLM-L6-v2` | 22M params, 384-dim | https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2 |
| Cross-encoder (reranker) | `cross-encoder/ms-marco-MiniLM-L6-v2` | 22M params | https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2 |
| Asymmetric embedder (prefix demo) | `intfloat/e5-small-v2` | 33M params, 384-dim | https://huggingface.co/intfloat/e5-small-v2 |
| LLM (generation) | `Qwen/Qwen2.5-0.5B-Instruct` (CPU) / `Qwen2.5-1.5B-Instruct` (GPU) | 0.5B / 1.5B | https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct |

> 💡 Want a bigger list? The **MTEB leaderboard** ranks embedding models: https://huggingface.co/spaces/mteb/leaderboard — but remember the lecture's rule: *the leaderboard builds the shortlist; your own data picks the winner.*

In [ ]:
EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L6-v2"   # = "...-L-6-v2" on the slides (renamed on the Hub)
E5_MODEL     = "intfloat/e5-small-v2"
LLM_MODEL    = "Qwen/Qwen2.5-1.5B-Instruct" if DEVICE == "cuda" else "Qwen/Qwen2.5-0.5B-Instruct"
print("LLM:", LLM_MODEL)

LLM: Qwen/Qwen2.5-1.5B-Instruct


---
# 1 · Our knowledge base: Acme Corp

A real RAG system would read files from a drive, a wiki or a data lake. To keep things self-contained, we **create** a few small company documents right here:

* 4 Markdown documents — HR policies (two subsidiaries: **Acme-India** and **Acme-US**), IT helpdesk, facilities, expenses.
* 1 PDF (the *Benefits Handbook*) — built in Section 4, with one **scanned** page to show a classic failure.

Notice that the two HR policies **look almost identical but have different numbers**. That's deliberate: it's where metadata filtering earns its keep.

In [ ]:
DOCS = {}

DOCS["hr_policy_india.md"] = """
# Acme-India HR Policy

## Annual leave
Employees accrue 18 days of paid leave annually, credited monthly at 1.5 days per month.
Unused leave of up to 30 days carries over into the next calendar year; anything above 30 days lapses on 31 March.

## Sick leave
Employees get 12 days of sick leave per year. A medical certificate is required for absences longer than 3 consecutive days.

## Parental leave
Birthing parents receive 26 weeks of paid maternity leave. Non-birthing parents receive 4 weeks of paid
paternity leave, to be taken within six months of the child's birth or adoption.

## Remote work
Employees may work from home up to 2 days per week with their manager's approval.
Fully remote arrangements need sign-off from the HR business partner.

## Notice period
The notice period is 60 days for permanent employees and 15 days during probation.
"""

DOCS["hr_policy_us.md"] = """
# Acme-US HR Policy

## Annual leave
Employees receive 15 days of paid time off (PTO) per year. PTO does not carry over; unused days are paid out in December.

## Sick leave
Employees get 10 days of sick leave per year. No doctor's note is needed for absences of up to 2 days.

## Parental leave
All new parents receive 12 weeks of paid parental leave, which can be split into two blocks.

## Remote work
Employees may work from home up to 3 days per week. Fully remote roles are listed as such in the job posting.
"""

DOCS["it_helpdesk.md"] = """
# Acme IT Helpdesk FAQ

## Error E1047: login token expired
This error appears when you log in to the employee portal after your single sign-on (SSO) session has expired.
Fix: sign out of all Acme apps, clear browser cookies for sso.acme.com, and log in again.
If it persists for more than an hour, raise a ticket with category 'Access'.

## Error E2001: disk quota exceeded
Your network home drive has reached its 50 GB limit. Delete old files or request a quota increase through the IT portal.

## Password reset
Passwords expire every 90 days. Reset yours at password.acme.com; you will need the authenticator app on your phone.

## VPN
Install the Acme VPN client from the Software Center. Use the 'India-West' gateway when working from Mumbai or Pune,
and the 'US-East' gateway when working from New York.
"""

DOCS["facilities.md"] = """
# Acme Facilities Guide

## Cafeteria
The cafeteria is open from 8 am to 6 pm on working days. Lunch is subsidised; breakfast and evening snacks are free.

## Parking
Two-wheeler parking is free. Car parking spots are allotted by lottery every quarter; apply through the facilities portal.

## Gym
The office gym on the 3rd floor is open from 6 am to 10 pm. Book a locker at the front desk.
"""

DOCS["expenses_policy.md"] = """
# Acme Travel and Expenses Policy

## Submitting claims
Submit travel expense claims within 30 days of returning from your trip, with itemised receipts for anything above INR 500.

## Per diem
The domestic per diem is INR 2,500 per day. International per diem depends on the city tier listed in the travel portal.

## Flights
Book economy class for flights under 6 hours. Premium economy is allowed for longer flights with VP approval.
"""

for name, text in DOCS.items():
    print(f"{name:<22} {len(text):>5} characters")

---
# 2 · Sparse retrieval with BM25 — and where it breaks

**Sparse retrieval** turns text into a giant vector with one slot per vocabulary word (mostly zeros) and scores documents by **shared keywords**. **BM25** is the classic scoring function: it rewards query words that appear often in a document but rarely across the corpus, and penalises very long documents.

$$\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t)\cdot \frac{f(t,d)\,(k_1+1)}{f(t,d) + k_1\left(1 - b + b\,\frac{|d|}{\text{avgdl}}\right)}$$

Let's reproduce the lecture's example: *"How much vacation do I get?"*

In [ ]:
from rank_bm25 import BM25Okapi

docs = ["Employees accrue 18 days of paid leave annually.",
        "Unused leave up to 30 days carries over to next year.",
        "The cafeteria is open from 8 am to 6 pm."]
query = "How much vacation do I get?"

def tokenize(text):
    """Very simple tokenizer: lowercase, keep letters/digits."""
    return re.findall(r"[a-z0-9]+", text.lower())

bm25 = BM25Okapi([tokenize(d) for d in docs])
print("Query tokens:", tokenize(query))
for d, s in zip(docs, bm25.get_scores(tokenize(query))):
    print(f"{s:6.3f}  {d}")

Query tokens: ['how', 'much', 'vacation', 'do', 'i', 'get']
 0.000  Employees accrue 18 days of paid leave annually.
 0.000  Unused leave up to 30 days carries over to next year.
 0.000  The cafeteria is open from 8 am to 6 pm.


**What to notice:** every score is **0**. The answer is in document 1, but it says *"paid leave"*, not *"vacation"*. BM25 matches **strings**, not **meaning** — this is the *vocabulary mismatch* problem (slide: *Problem with Sparse Retrieval*).

But BM25 is great at **exact rare tokens** like error codes — keep that in mind for Section 7:

In [ ]:
it_docs = ["Error E1047: login token expired. Clear SSO cookies and log in again.",
           "Error E2001: disk quota exceeded on your home drive.",
           "How to log in to the employee portal for the first time."]
bm25_it = BM25Okapi([tokenize(d) for d in it_docs])
for d, s in zip(it_docs, bm25_it.get_scores(tokenize("I get E1047"))):
    print(f"{s:6.3f}  {d}")

 0.491  Error E1047: login token expired. Clear SSO cookies and log in again.
 0.000  Error E2001: disk quota exceeded on your home drive.
 0.000  How to log in to the employee portal for the first time.


🔗 **Read more**
- `rank_bm25` library: https://github.com/dorianbrown/rank_bm25
- Robertson & Zaragoza, *The Probabilistic Relevance Framework: BM25 and Beyond* (2009): https://www.nowpublishers.com/article/Details/INR-019
- Production BM25: Elasticsearch / OpenSearch, or `bm25s` (fast pure-Python): https://github.com/xhluca/bm25s

---
# 3 · Dense embeddings

A **dense embedding** maps a whole sentence to a short vector (384 numbers for MiniLM) where **geometry = meaning**: similar sentences point in similar directions. We compare them with **cosine similarity**:

$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a}\cdot\mathbf{b}}{\|\mathbf{a}\|\,\|\mathbf{b}\|}\in[-1, 1]$$

If we **normalise** every vector to length 1, cosine similarity is just the dot product `a @ b`. We'll use that trick everywhere.

### 3.1 Load a bi-encoder and embed some sentences

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
print("Embedding dimension:", embedder.get_sentence_embedding_dimension())
print("Max input length (tokens):", embedder.max_seq_length)   # longer text is silently cut off!

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384
Max input length (tokens): 256


In [ ]:
doc_vecs = embedder.encode(docs, normalize_embeddings=True)      # shape (3, 384)
q_vec    = embedder.encode(query, normalize_embeddings=True)     # shape (384,)

print("doc_vecs shape:", doc_vecs.shape)
print("First 8 numbers of the query vector:", q_vec[:8].round(3))
print("Vector length (norm):", np.linalg.norm(q_vec).round(3))

print("\nCosine similarity to:", query)
for d, s in zip(docs, doc_vecs @ q_vec):
    print(f"{s:6.3f}  {d}")

**What to notice:** BM25 gave all zeros; the dense model ranks the **paid-leave** sentences far above the cafeteria one — even with **zero shared words**. That's why dense retrieval exists.

### 3.2 Feel the geometry: a similarity heat-map

In [ ]:
import matplotlib.pyplot as plt

sentences = ["How much vacation do I get?",
             "Employees accrue 18 days of paid leave annually.",
             "I can't log in to the portal.",
             "Error E1047: login token expired.",
             "When does the canteen open?",
             "The cafeteria is open from 8 am to 6 pm."]
E = embedder.encode(sentences, normalize_embeddings=True)
S = E @ E.T

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(S, cmap="Blues", vmin=0, vmax=1)
short = [s[:28] + "…" if len(s) > 28 else s for s in sentences]
ax.set_xticks(range(len(short)), short, rotation=45, ha="right"); ax.set_yticks(range(len(short)), short)
for i in range(len(S)):
    for j in range(len(S)):
        ax.text(j, i, f"{S[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if S[i, j] > 0.6 else "black")
plt.colorbar(im); plt.title("Cosine similarity (all-MiniLM-L6-v2)"); plt.tight_layout(); plt.show()

**What to notice:** the question/answer pairs (vacation ↔ paid leave, canteen ↔ cafeteria) light up even though the words differ.

### 3.3 ⏩ OPTIONAL — Why not just use raw BERT? ([CLS] vs mean-pooling)

BERT outputs **one vector per token**. To get one sentence vector you either take the `[CLS]` token or **average** (mean-pool) all tokens. But raw BERT was **never trained** to make these vectors comparable with cosine — SBERT fine-tunes it so that similar sentences land close together. Let's see it (downloads ~440 MB).

In [ ]:
from transformers import AutoTokenizer, AutoModel

bert_tok = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
bert = AutoModel.from_pretrained("google-bert/bert-base-uncased").to(DEVICE).eval()

@torch.no_grad()
def bert_embed(texts, pooling="mean"):
    batch = bert_tok(texts, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
    out = bert(**batch).last_hidden_state                       # (batch, tokens, 768)
    if pooling == "cls":
        v = out[:, 0]                                           # vector of the [CLS] token
    else:
        mask = batch["attention_mask"].unsqueeze(-1).float()    # ignore padding tokens
        v = (out * mask).sum(1) / mask.sum(1)                   # mean over real tokens
    return torch.nn.functional.normalize(v, dim=-1).cpu().numpy()

for pooling in ["cls", "mean"]:
    Eb = bert_embed(docs + [query], pooling)
    print(f"raw BERT [{pooling}]  query vs docs:", (Eb[:3] @ Eb[3]).round(2))
print("SBERT (MiniLM)       query vs docs:", (doc_vecs @ q_vec).round(2))

**What to notice:** raw BERT gives *every* document a similar, high score — it can barely tell the cafeteria sentence apart from the leave sentences. SBERT spreads the scores out so ranking actually works.

### 3.4 Bi-encoder vs cross-encoder

| | **Bi-encoder** | **Cross-encoder** |
|---|---|---|
| Input | query and document encoded **separately** | query **and** document encoded **together** |
| Document vectors | pre-computed once, stored in an index | nothing can be pre-computed |
| Cost per query over 1M chunks | 1 forward pass (≈ 5 ms) | 1M forward passes (≈ 83 min!) |
| Accuracy | good | better |
| Use it for | **retrieval** (top 100 from millions) | **reranking** (top 100 → top 5) |

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(RERANK_MODEL, device=DEVICE)

pairs = [(query, d) for d in docs]
ce_scores = reranker.predict(pairs)
for d, s in zip(docs, ce_scores):
    print(f"{s:8.2f}  {d}")
print("\n(Cross-encoder scores are raw relevance logits — only the ORDER matters, not the scale.)")

In [ ]:
# Quiz 1 from the slides, as code: cost of scoring 1,000,000 chunks for ONE query
N_CHUNKS, MS_PER_PASS = 1_000_000, 5
print(f"Bi-encoder   : 1 pass (only the query)  = {MS_PER_PASS} ms")
print(f"Cross-encoder: {N_CHUNKS:,} passes = {N_CHUNKS * MS_PER_PASS / 1000 / 60:.0f} minutes")
print(f"Cross-encoder on a top-100 shortlist = {100 * MS_PER_PASS / 1000:.1f} s   ← why we only RERANK with it")

### 3.5 The #1 silent bug: missing `query:` / `passage:` prefixes

Many modern embedders (E5, BGE, GTE, Qwen3-Embedding, Instructor, Nomic…) are **asymmetric**: they were trained with a *different prefix* for queries and for documents. Forget the prefix and recall drops — **with no error message**. Check the model card! For E5: https://huggingface.co/intfloat/e5-small-v2

In [ ]:
e5 = SentenceTransformer(E5_MODEL, device=DEVICE)

passages = ["Employees accrue 18 days of paid leave annually.",
            "How much vacation do other companies give? A survey of 500 firms.",
            "The cafeteria is open from 8 am to 6 pm."]

def e5_scores(q, ps, use_prefix):
    if use_prefix:
        q, ps = "query: " + q, ["passage: " + p for p in ps]
    qv = e5.encode(q, normalize_embeddings=True)
    pv = e5.encode(ps, normalize_embeddings=True)
    return (pv @ qv).round(3)

print("WITHOUT prefixes:", e5_scores(query, passages, use_prefix=False))
print("WITH prefixes   :", e5_scores(query, passages, use_prefix=True))

**What to notice:** look at how the gap between the right passage (#1) and the distractor (#2) changes. Without prefixes the model treats your question like just another document, so passages that *look like* the question (#2 repeats its words) get pulled up. *(Exact numbers vary by model version — the habit to build is: **read the model card, add the prefixes**.)*

> In `sentence-transformers` ≥ v5 you can also use `model.encode_query(...)` / `model.encode_document(...)`, which apply the model's prompts for you if the model card defines them.

### 3.6 How big will my index be? (Matryoshka + float16)

Storage = **#vectors × dims × bytes per number**. **Matryoshka (MRL)** models (OpenAI `text-embedding-3-*`, Nomic, Gemini) are trained so that the *first k dimensions* are already a usable embedding — you can just slice the vector. **Never** slice a non-MRL model like MiniLM.

In [ ]:
def index_size_gb(n_vectors, dims, bytes_per_value=4):
    return n_vectors * dims * bytes_per_value / 1e9

N = 100_000_000
for name, d in [("all-MiniLM-L6-v2", 384), ("bge-base-en-v1.5", 768),
                ("text-embedding-3-small", 1536), ("text-embedding-3-large", 3072)]:
    print(f"{name:<24} {d:>5} dims  →  {index_size_gb(N, d):>7.0f} GB for 100M vectors (float32)")

# Quiz 2 from the slides: MRL-truncate 3072 → 512 dims AND store float16 (2 bytes)
print(f"\n3072-d float32: {index_size_gb(N, 3072):.0f} GB   →   512-d float16: {index_size_gb(N, 512, 2):.1f} GB"
      f"   ({index_size_gb(N, 3072) / index_size_gb(N, 512, 2):.0f}× smaller)")

🔗 **Read more — embeddings**
- Sentence-Transformers docs: https://sbert.net · pretrained models: https://sbert.net/docs/sentence_transformer/pretrained_models.html · cross-encoders: https://sbert.net/docs/cross_encoder/pretrained_models.html
- Reimers & Gurevych, *Sentence-BERT* (2019): https://arxiv.org/abs/1908.10084
- Karpukhin et al., *Dense Passage Retrieval (DPR)* (2020): https://arxiv.org/abs/2004.04906
- Wang et al., *E5: Text Embeddings by Weakly-Supervised Contrastive Pre-training* (2022): https://arxiv.org/abs/2212.03533
- Kusupati et al., *Matryoshka Representation Learning* (2022): https://arxiv.org/abs/2205.13147 · HF blog: https://huggingface.co/blog/matryoshka
- Wang et al., *MiniLMv2* (2020): https://arxiv.org/abs/2012.15828
- MTEB leaderboard: https://huggingface.co/spaces/mteb/leaderboard
- BGE-M3 (multilingual, 8K tokens): https://huggingface.co/BAAI/bge-m3 · Qwen3-Embedding: https://huggingface.co/Qwen/Qwen3-Embedding-0.6B · multilingual-e5-large: https://huggingface.co/intfloat/multilingual-e5-large
- OpenAI embeddings guide: https://platform.openai.com/docs/guides/embeddings
- Visual intro to sentence embeddings (lecture source): https://github.com/hkproj/retrieval-augmented-generation-notes

---
# 4 · Document processing: PDFs and the scanned-page trap

> *Garbage in, garbage out.* Retrieval quality is capped by how cleanly you extract text before chunking.

We now create the **Acme Benefits Handbook** PDF with 3 pages:
- pages 1–2: normal **digital** text,
- page 3: a **scanned image** of text — it *looks* like text to a human but has **no text layer**.

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib.utils import ImageReader
from PIL import Image, ImageDraw, ImageFont

PDF_PATH = "benefits_handbook.pdf"

PAGE_TEXT = {
    1: ["Acme Benefits Handbook 2026",
        "",
        "Health insurance",
        "All employees and their dependants (spouse, up to two children and parents)",
        "are covered for up to INR 5,00,000 per year under the group health plan.",
        "Claims are cashless at network hospitals; others are reimbursed within 21 days."],
    2: ["Wellness and learning",
        "",
        "An annual health check-up is free for every employee above the age of 30.",
        "Gym memberships are reimbursed up to INR 1,500 per month.",
        "Every employee has a learning budget of INR 50,000 per year for courses and",
        "certifications. Manager approval is needed before you enrol."],
    3: ["Employee stock options",
        "",
        "Stock options vest 25% after the first year of service,",
        "then monthly over the following three years.",
        "Unvested options lapse on your last working day."],
}

def scanned_image(lines):
    """Render text into a PNG, like a photocopy: pixels only, no text layer."""
    img = Image.new("RGB", (1240, 700), "white")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 34)
    except OSError:
        font = ImageFont.load_default()
    for i, line in enumerate(lines):
        draw.text((60, 60 + i * 60), line, fill="black", font=font)
    return img

c = canvas.Canvas(PDF_PATH, pagesize=A4)
W, H = A4
for page_no, lines in PAGE_TEXT.items():
    if page_no < 3:                                   # digital page → real text layer
        y = H - 80
        for line in lines:
            c.setFont("Helvetica-Bold" if line == lines[0] else "Helvetica", 13)
            c.drawString(60, y, line); y -= 22
    else:                                             # "scanned" page → just a picture
        c.drawImage(ImageReader(scanned_image(lines)), 40, H - 420, width=W - 80, height=(W - 80) * 700 / 1240)
    c.showPage()
c.save()
print("Created", PDF_PATH)

In [ ]:
import pypdf

reader = pypdf.PdfReader(PDF_PATH)
for i, page in enumerate(reader.pages, start=1):
    text = (page.extract_text() or "").strip()
    print(f"--- page {i}: {len(text)} characters extracted")
    print(text[:200] if text else "⚠️  (empty — this page has no text layer!)")

**What to notice:** page 3 returns an **empty string** — no error, no warning. If you embed it, your index silently contains nothing about stock options.

### The slide's "probe" — and its trap (Quiz 4)
The lecture showed a quick probe that samples the first few pages. Watch what happens when only *some* pages are scanned:

In [ ]:
def pdf_has_text_layer(path, sample=3):
    r = pypdf.PdfReader(path)
    return any((p.extract_text() or "").strip() for p in r.pages[:sample])

print("Probe says the PDF has a text layer:", pdf_has_text_layer(PDF_PATH))
print("→ so the whole file goes to pypdf, and page 3 is silently lost.")
print("✅ Fix: decide page by page (next cell).")

In [ ]:
def extract_pages(path, use_ocr=False):
    """Text for every page. Flags pages with no text layer; OCRs them if use_ocr=True."""
    pages = []
    for i, page in enumerate(pypdf.PdfReader(path).pages, start=1):
        text = (page.extract_text() or "").strip()
        needs_ocr = not text
        if needs_ocr and use_ocr:
            import pytesseract
            from pdf2image import convert_from_path
            img = convert_from_path(path, dpi=300, first_page=i, last_page=i)[0]
            text = pytesseract.image_to_string(img).strip()
        pages.append({"page": i, "text": text, "needs_ocr": needs_ocr})
    return pages

pdf_pages = extract_pages(PDF_PATH, use_ocr=False)
print(sum(p["needs_ocr"] for p in pdf_pages), "of", len(pdf_pages), "pages need OCR")

### ⏩ OPTIONAL — Run OCR on the scanned page

OCR = *optical character recognition*. We use **Tesseract** (free, local). For hard scans, tables and multi-column layouts, use a layout-aware parser or a vision model instead (links below).

In [ ]:
RUN_OCR = False          # ← set to True to try OCR (installs Tesseract + Poppler, ~1 min on Colab)

if RUN_OCR:
    !apt-get -qq install -y tesseract-ocr poppler-utils > /dev/null
    !pip install -q pytesseract pdf2image
    pdf_pages = extract_pages(PDF_PATH, use_ocr=True)
    print("Page 3 after OCR:\n", pdf_pages[2]["text"])
else:
    print("Skipping OCR. Page 3 stays empty — remember this when we evaluate retrieval in Section 10!")

### Which parser for which document? (cheat-sheet from the slides)

| Document | Use | Why |
|---|---|---|
| Plain-prose PDF | `pypdf` | fast and light; nothing to preserve |
| PDF whose value is in **tables** | `pdfplumber` / `Camelot` → serialise tables to Markdown/HTML | `pypdf` flattens tables into word salad |
| **Scanned** PDF | OCR: `tesseract` + `pdf2image`, or a cloud OCR / vision model | there is literally no text to extract |
| Multi-column / forms / papers | layout-aware: `docling`, `Unstructured`, `GROBID`, Azure Document Intelligence | `pypdf` interleaves columns |
| HTML you control | `BeautifulSoup` with targeted selectors | precise |
| HTML from the open web | `trafilatura` | strips nav, ads, cookie banners |
| Big tables / databases | **Text-to-SQL**, not embeddings | "top 5 customers by revenue" is a computation, not a similarity search |

🔗 **Read more — document processing**
- pypdf: https://pypdf.readthedocs.io · pdfplumber: https://github.com/jsvine/pdfplumber · Camelot: https://camelot-py.readthedocs.io
- Tesseract: https://github.com/tesseract-ocr/tesseract · pytesseract: https://github.com/madmaze/pytesseract · pdf2image: https://github.com/Belval/pdf2image
- docling (IBM): https://github.com/docling-project/docling · Unstructured: https://docs.unstructured.io · GROBID: https://github.com/kermitt2/grobid
- trafilatura: https://trafilatura.readthedocs.io · BeautifulSoup: https://www.crummy.com/software/BeautifulSoup/bs4/doc/
- LangChain document loaders (PDF, web, Notion, Drive, …): https://docs.langchain.com/oss/python/integrations/document_loaders

---
# 5 · Chunking

We don't embed whole documents: **one chunk = one vector**, and the chunk is the true **unit of retrieval**.
- Too **small** → loses context (*"it rose 40%"* — what rose?).
- Too **big** → the vector gets diluted, and the model may silently truncate it (MiniLM reads at most 256 tokens!).

| Strategy | How it splits | Use it for |
|---|---|---|
| Fixed-size | every N tokens | baseline |
| **Sliding window** | N tokens with ~10–20% overlap | most RAG (**the default**) |
| Structure-aware | headings, clauses, functions | Markdown, legal, API docs, code |
| Semantic | where the meaning shifts | topic-shifting documents |

In [ ]:
text = DOCS["hr_policy_india.md"]

def fixed_char_chunks(text, size=200):
    return [text[i:i + size] for i in range(0, len(text), size)]

for i, ch in enumerate(fixed_char_chunks(text)[:3]):
    print(f"[chunk {i}] …{ch!r}…\n")

**What to notice:** chunks start and end **mid-word, mid-sentence**. A fact like *"anything above 30 days lapses on 31 March"* can be cut in two, so neither half retrieves well.

### 5.2 Sliding window by **tokens** (the lecture's default)

Count length in the **embedding model's own tokens** (its limit is in tokens, not characters), then slice the *original* text. The overlap ensures a fact that straddles a boundary lands intact in at least one chunk.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(EMBED_MODEL)

def sliding_window(text, size=64, overlap=16):
    enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
    spans = enc["offset_mapping"]                 # (start_char, end_char) of every token
    step = size - overlap
    chunks = []
    for s in range(0, max(len(spans) - overlap, 1), step):
        e = min(s + size, len(spans)) - 1
        chunks.append(text[spans[s][0]:spans[e][1]])   # slice the ORIGINAL text
    return chunks

print("Total tokens in the India HR policy:", len(tok(text, add_special_tokens=False)["input_ids"]))
windows = sliding_window(text, size=64, overlap=16)
print(len(windows), "chunks\n")
for i, ch in enumerate(windows[:3]):
    print(f"[chunk {i}] {ch!r}\n")

*(We use tiny 64-token windows so you can see the overlap on screen. A sensible real-world start is ~256–512 tokens with ~20% overlap — but check your embedder's `max_seq_length` first.)*

### 5.3 Recursive splitting (LangChain)

`RecursiveCharacterTextSplitter` tries to split on **paragraphs** first, then **lines**, then **sentences/words**, and only falls back to raw characters if a piece is still too long. It respects natural boundaries — the most popular splitter in practice. We can make it count **tokens** of our embedding model too.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tok, chunk_size=64, chunk_overlap=10,
    separators=["\n\n", "\n", ". ", " ", ""])     # tried in this order

for i, ch in enumerate(splitter.split_text(text)[:4]):
    print(f"[chunk {i}] {ch!r}\n")

### 5.4 Structure-aware: split on Markdown headings ✅ (what we'll use)

Our documents already tell us where the seams are: each `##` heading is one self-contained answer. We split on headings **and keep the heading as metadata** — useful for filtering and citing later.

**Pro tip — contextual chunk headers:** prepend the document title and section to the chunk text before embedding. *"Annual leave: 18 days"* alone is ambiguous across subsidiaries; *"Acme-India HR Policy › Annual leave: 18 days"* is not.

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "title"), ("##", "section")])
company_of = {"hr_policy_india.md": "Acme-India", "hr_policy_us.md": "Acme-US"}

chunks = []      # ← our knowledge base: a list of dicts {id, text, metadata}
for source, doc_text in DOCS.items():
    for j, piece in enumerate(md_splitter.split_text(doc_text)):
        title, section = piece.metadata.get("title", ""), piece.metadata.get("section", "")
        if not section:                      # skip the bare title line
            continue
        chunks.append({
            "id": f"{source}#{j}",
            "text": f"{title} › {section}: {piece.page_content.strip()}",   # contextual header
            "metadata": {"source": source, "section": section,
                         "company": company_of.get(source, "Acme (all)"), "page": 0},
        })

# PDF pages: one chunk per page (they're short); split further if a page is long
for p in pdf_pages:
    for j, piece in enumerate(splitter.split_text(p["text"]) if p["text"] else [""]):
        chunks.append({
            "id": f"{PDF_PATH}#p{p['page']}-{j}",
            "text": f"Acme Benefits Handbook › page {p['page']}: {piece}",
            "metadata": {"source": PDF_PATH, "section": f"page {p['page']}",
                         "company": "Acme (all)", "page": p["page"]},
        })

print(len(chunks), "chunks in the knowledge base\n")
for ch in chunks[:3] + chunks[-2:]:
    print(ch["id"], "|", ch["metadata"])
    show(ch["text"]); print()

**What to notice:** the last chunk (PDF page 3) contains *only the header* — the empty scanned page made it into the knowledge base. In production you'd drop or flag it; we keep it to show the effect in Section 10.

### 5.5 ⏩ OPTIONAL — Semantic chunking

Split where the **meaning shifts**: embed every sentence and cut where the similarity between neighbouring sentences drops the most (here: the lowest 30% of neighbour similarities — the same "percentile" idea LangChain's `SemanticChunker` uses). Good for long, topic-shifting documents — but you embed every sentence at ingest, and you should always cap the chunk size.

In [ ]:
def semantic_chunks(text, percentile=30, max_sentences=6):
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.replace("\n", " ")) if s.strip()]
    vecs = embedder.encode(sents, normalize_embeddings=True)
    sims = (vecs[:-1] * vecs[1:]).sum(axis=1)          # similarity of sentence i and i+1
    threshold = np.percentile(sims, percentile)        # cut at the biggest drops (bottom 30%)
    out, current = [], [sents[0]]
    for s, sim in zip(sents[1:], sims):
        if sim < threshold or len(current) >= max_sentences:   # meaning shift (or size cap)
            out.append(" ".join(current)); current = []
        current.append(s)
    out.append(" ".join(current))
    return out, sims

# Concatenate two unrelated docs with NO headings, and let the model find the seam
mixed = re.sub(r"#+ .*\n", "", DOCS["facilities.md"] + DOCS["expenses_policy.md"])
sem, sims = semantic_chunks(mixed)
print("Neighbour similarities:", sims.round(2))
for i, ch in enumerate(sem):
    print(f"\n[semantic chunk {i}]"); show(ch)

🔗 **Read more — chunking**
- LangChain text splitters: https://docs.langchain.com/oss/python/integrations/splitters
- Pinecone, *Chunking strategies for LLM applications*: https://www.pinecone.io/learn/chunking-strategies/
- Greg Kamradt, *5 Levels of Text Splitting* (notebook): https://github.com/FullStackRetrieval-com/RetrievalTutorials/blob/main/tutorials/LevelsOfTextSplitting/5_Levels_Of_Text_Splitting.ipynb
- Anthropic, *Contextual Retrieval* (contextual chunk headers, done with an LLM): https://www.anthropic.com/news/contextual-retrieval
- LlamaIndex node parsers (another popular framework): https://docs.llamaindex.ai

---
# 6 · Vector stores: FAISS and ChromaDB

![semantic_search.jpg](attachment:semantic_search.jpg)

First, embed every chunk **once** (offline, at index time). This is the expensive part we never repeat per query.

In [ ]:
chunk_texts = [c["text"] for c in chunks]
chunk_vecs = embedder.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=False).astype("float32")
print("Chunk matrix:", chunk_vecs.shape, "| memory:", chunk_vecs.nbytes / 1024, "KB")

### 6.1 FAISS — exact search with `IndexFlatIP`

`IndexFlatIP` = brute-force **I**nner **P**roduct. On **normalised** vectors, inner product = cosine. It checks every vector — exact, no training — perfect up to ~1M vectors.

In [ ]:
import faiss

dim = chunk_vecs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_vecs)
print("Vectors in index:", index.ntotal)

def faiss_search(question, k=3):
    qv = embedder.encode([question], normalize_embeddings=True).astype("float32")   # SAME model!
    scores, ids = index.search(qv, k)
    return [(chunks[i], float(s)) for i, s in zip(ids[0], scores[0])]

for ch, s in faiss_search("How much vacation do I get?"):
    print(f"{s:.3f}  {ch['id']:<24} {ch['text'][:80]}…")

**What to notice:**
1. FAISS returns only **numbers** — row ids and scores. *We* keep the `id → text/metadata` mapping (the `chunks` list). Lose it and you have the classic *"numbers but no documents"* bug.
2. India **and** US leave policies come back. Which one is right depends on who's asking → we need **metadata filtering** (6.4).

### 6.2 The metric trap: normalise before `IndexFlatIP`

In [ ]:
raw = embedder.encode(chunk_texts, normalize_embeddings=False).astype("float32")    # NOT normalised
qv_raw = embedder.encode(["How much vacation do I get?"], normalize_embeddings=False).astype("float32")
print("Vector norms vary:", np.linalg.norm(raw, axis=1)[:5].round(2), "...")

ip = faiss.IndexFlatIP(dim); ip.add(raw)
print("Top-3 with raw inner product:", ip.search(qv_raw, 3)[1][0])
print("Top-3 with cosine           :", index.search(qv_raw / np.linalg.norm(qv_raw), 3)[1][0])
print("(If the lists differ, longer vectors 'won' just for being long. Use the metric the model was trained with.)")

*(MiniLM's raw vectors happen to be close to unit length, so the lists may match here — with other models they often don't. The habit: `normalize_embeddings=True` or `faiss.normalize_L2(x)` before an IP index.)*

### 6.3 ⏩ OPTIONAL — Approximate search at scale: IVF, PQ and HNSW

Our 30 chunks are too few to need approximate search, so we use 100,000 synthetic "clustered" vectors (same code as the slides) and measure **recall@10** against the exact answer.

- **IVF** (inverted file): k-means into `nlist ≈ √N` buckets; search only the `nprobe` nearest buckets → scans ~N·nprobe/nlist vectors.
- **PQ** (product quantisation): split each vector into `m` sub-vectors, replace each by a 1-byte centroid id → 1536 bytes become `m` bytes.
- **HNSW**: a navigable graph; very fast at high recall, but uses more RAM. It's what ChromaDB uses under the hood.

In [ ]:
rng, d, n = np.random.default_rng(0), 384, 100_000
topics = rng.normal(size=(100, d))
xb = (topics[rng.integers(0, 100, n)] + 3 * rng.normal(size=(n, d))).astype("float32")
faiss.normalize_L2(xb)
xq = xb[:200].copy()

exact = faiss.IndexFlatIP(d); exact.add(xb)
t0 = time.perf_counter(); _, truth = exact.search(xq, 10); t_exact = time.perf_counter() - t0

def report(name, idx, extra=""):
    t0 = time.perf_counter(); _, found = idx.search(xq, 10); t = time.perf_counter() - t0
    recall = np.mean([len(set(a) & set(b)) / 10 for a, b in zip(truth, found)])
    print(f"{name:<22} recall@10={recall:.2f}   {1000 * t / len(xq):.3f} ms/query   {extra}")

print(f"{'Flat (exact)':<22} recall@10=1.00   {1000 * t_exact / len(xq):.3f} ms/query   {d * 4} bytes/vector")

nlist = int(math.sqrt(n))                                       # ≈ 316 buckets
ivf = faiss.IndexIVFFlat(faiss.IndexFlatIP(d), d, nlist, faiss.METRIC_INNER_PRODUCT)
ivf.train(xb); ivf.add(xb)
for nprobe in (1, 8, 32):
    ivf.nprobe = nprobe
    report(f"IVF  nprobe={nprobe}", ivf, f"scans ≈ {n * nprobe // nlist:,} of {n:,}")

In [ ]:
# IVF + PQ: m sub-vectors × 8 bits → m bytes per vector instead of 1536
for m in (8, 48, 96):
    ivfpq = faiss.IndexIVFPQ(faiss.IndexFlatIP(d), d, nlist, m, 8, faiss.METRIC_INNER_PRODUCT)
    ivfpq.train(xb); ivfpq.add(xb); ivfpq.nprobe = 32
    report(f"IVF-PQ m={m} nprobe=32", ivfpq, f"{m} bytes/vector  ({d * 4 // m}× smaller)")

# HNSW graph index; efSearch is its recall/speed dial
hnsw = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT)
hnsw.add(xb)
for ef in (16, 64, 256):
    hnsw.hnsw.efSearch = ef
    report(f"HNSW efSearch={ef}", hnsw)

**What to notice:** every approximate index trades **recall** for **speed** or **memory**. PQ is **lossy**: at 8 bytes/vector (192× smaller) recall collapses on this deliberately noisy data, and more sub-vectors buy it back. In production, PQ is usually paired with a second pass that re-scores the shortlist with exact vectors (FAISS `IndexRefineFlat`) or with a cross-encoder. Rule of thumb from the slides:

| Corpus | Index |
|---|---|
| ≤ ~1M vectors, or you need 100% recall | `IndexFlatIP` |
| Millions, fits in RAM | `IndexIVFFlat` (`nlist ≈ √N`, tune `nprobe`) or `IndexHNSWFlat` |
| Huge, memory-constrained | `IndexIVFPQ` |

### 6.4 ChromaDB — a vector **database**: vectors + text + metadata + filters

FAISS is a *library* that finds nearest vectors. Chroma wraps that kind of search in a *database*: it stores the text and metadata next to each vector, persists to disk, and can **filter by metadata before searching**.

In [ ]:
import chromadb

client = chromadb.EphemeralClient()            # in-memory; use chromadb.PersistentClient(path="./db") to save to disk
try:
    client.delete_collection("acme")            # lets you re-run this cell safely
except Exception:
    pass
collection = client.create_collection("acme", metadata={"hnsw:space": "cosine"})   # match the model's metric!

collection.add(
    ids=[c["id"] for c in chunks],
    embeddings=chunk_vecs,                      # we pass OUR vectors (same model we'll use for queries)
    documents=[c["text"] for c in chunks],
    metadatas=[c["metadata"] for c in chunks],
)
print("Records in Chroma:", collection.count())

In [ ]:
def chroma_search(question, k=3, where=None):
    qv = embedder.encode([question], normalize_embeddings=True)
    res = collection.query(query_embeddings=qv, n_results=k, where=where,
                           include=["documents", "metadatas", "distances"])
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        print(f"cos={1 - dist:.3f}  [{meta['company']:<10} | {meta['source']}]  {doc[:70]}…")

q = "How many days of leave do I get per year?"
print("No filter:"); chroma_search(q)
print("\nPre-filtered to Acme-US:"); chroma_search(q, where={"company": "Acme-US"})
print("\nFilter with operators ($in, $ne, $and …):")
chroma_search(q, where={"company": {"$in": ["Acme-India", "Acme (all)"]}})

*(Chroma returns **distance** = 1 − cosine similarity, so smaller is better. We convert it back for readability.)*

### 6.5 ⏩ OPTIONAL — Why **pre**-filter? (Quiz 8 as code)

50 subsidiaries each have a near-identical leave policy. If you fetch the top-10 and **then** filter to your subsidiary, you'll often miss it.

In [ ]:
countries = ["India", "US", "UK", "Germany", "France", "Japan", "Brazil", "Kenya", "Canada", "Mexico"]
subs = [f"Acme-{c}-{i}" for i in range(5) for c in countries]            # 50 subsidiaries
sub_texts = [f"{s} HR Policy › Annual leave: Employees receive {15 + i % 11} days of paid leave per year."
             for i, s in enumerate(subs)]
sub_vecs = embedder.encode(sub_texts, normalize_embeddings=True)

try: client.delete_collection("subsidiaries")
except Exception: pass
subcol = client.create_collection("subsidiaries", metadata={"hnsw:space": "cosine"})
subcol.add(ids=subs, embeddings=sub_vecs, documents=sub_texts, metadatas=[{"company": s} for s in subs])

qv = embedder.encode(["How many leave days do I get?"], normalize_embeddings=True)
post_hits = pre_hits = 0
for target in subs:
    top10 = subcol.query(query_embeddings=qv, n_results=10)["ids"][0]                    # search, THEN filter
    post_hits += target in top10
    pre = subcol.query(query_embeddings=qv, n_results=1, where={"company": target})["ids"][0]   # filter, THEN search
    pre_hits += target in pre
print(f"Post-filter (top-10, then filter): found the right policy {post_hits}/50 times")
print(f"Pre-filter  (where=...)          : found the right policy {pre_hits}/50 times")

🔗 **Read more — vector stores**
- FAISS wiki: https://github.com/facebookresearch/faiss/wiki · *Guidelines to choose an index*: https://github.com/facebookresearch/faiss/wiki/Guidelines-to-choose-an-index · index list: https://github.com/facebookresearch/faiss/wiki/Faiss-indexes
- Douze et al., *The Faiss library* (2024): https://arxiv.org/abs/2401.08281 · Johnson et al., *Billion-scale similarity search with GPUs* (2017): https://arxiv.org/abs/1702.08734
- Jégou et al., *Product Quantization for Nearest Neighbor Search* (2011): https://ieeexplore.ieee.org/document/5432202
- Malkov & Yashunin, *HNSW* (2016): https://arxiv.org/abs/1603.09320
- Pinecone's free FAISS course (IVF, PQ, HNSW with pictures): https://www.pinecone.io/learn/series/faiss/
- ChromaDB docs: https://docs.trychroma.com (see *Querying* and *Metadata filtering*)
- Other vector DBs: Qdrant https://qdrant.tech/documentation/ · Weaviate https://weaviate.io/developers/weaviate · Milvus https://milvus.io/docs · Pinecone https://docs.pinecone.io · pgvector https://github.com/pgvector/pgvector
- Video on IVF (from the slides): https://www.youtube.com/watch?v=BMYBwbkbVec

---
# 7 · Hybrid search with Reciprocal Rank Fusion (RRF)

Dense retrieval matches **meaning**; BM25 matches **exact tokens** (error codes, SKUs, function names). Run both, then **fuse the rankings**. BM25 and cosine scores live on different scales, so we fuse **ranks**, not scores:

$$\text{RRF}(d) = \sum_{\text{lists}} \frac{1}{k + \text{rank}(d)}, \qquad k \approx 60$$

In [ ]:
def rrf(rankings, k=60):
    """Reciprocal Rank Fusion: score(d) = sum over lists of 1 / (k + rank)."""
    fused = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            fused[doc] = fused.get(doc, 0) + 1 / (k + rank)
    return sorted(fused.items(), key=lambda kv: kv[1], reverse=True)

# The slide example
bm25_rank  = ["E1047-fix", "E2001-disk", "login-faq"]
dense_rank = ["login-faq", "E1047-fix", "E2001-disk"]
for doc, score in rrf([bm25_rank, dense_rank]):
    print(f"{doc:<11} {score:.4f}")

Now on our real knowledge base: one BM25 index + our FAISS dense index, fused with RRF.

In [ ]:
bm25_kb = BM25Okapi([tokenize(c["text"]) for c in chunks])

def bm25_ranking(question, k=10):
    scores = bm25_kb.get_scores(tokenize(question))
    return [int(i) for i in np.argsort(-scores)[:k]]                 # chunk indices, best first

def dense_ranking(question, k=10):
    qv = embedder.encode([question], normalize_embeddings=True).astype("float32")
    return [int(i) for i in index.search(qv, k)[1][0]]

def hybrid_ranking(question, k=10):
    return [i for i, _ in rrf([bm25_ranking(question, k), dense_ranking(question, k)])][:k]

def compare(question, k=3):
    print("Q:", question)
    for name, fn in [("BM25", bm25_ranking), ("Dense", dense_ranking), ("Hybrid", hybrid_ranking)]:
        print(f"  {name:<6} →", [chunks[i]["metadata"]["section"][:28] for i in fn(question, k)])
    print()

compare("error E1047 when I log in")
compare("When does the canteen open?")
compare("How much vacation do I get?")

**What to notice:** BM25 nails the exact code `E1047`; dense nails the paraphrases (canteen ↔ cafeteria, vacation ↔ leave); hybrid gets the best of both. *"Consistent in both lists beats spiky in one."*

🔗 **Read more**
- Cormack, Clarke & Büttcher, *Reciprocal Rank Fusion* (SIGIR 2009): https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf
- Hybrid search in Weaviate / Qdrant: https://weaviate.io/blog/hybrid-search-explained · https://qdrant.tech/articles/hybrid-search/

---
# 8 · Rerank + generate: the full pipeline

The industry-standard **two-stage** pattern:

1. **Retrieve** — bi-encoder (+ BM25) pulls a generous shortlist (top 20 here; 100–200 in production).
2. **Rerank** — the cross-encoder rescores each *(question, chunk)* pair jointly and keeps the best few.
3. **Generate** — the top chunks go into the prompt, and the LLM answers **only from them**, with citations.

In [ ]:
def retrieve(question, k=20, company=None):
    """Stage 1: hybrid retrieval, with an optional metadata pre-filter on company."""
    candidates = hybrid_ranking(question, k=len(chunks))          # rank everything (tiny corpus)
    if company:
        candidates = [i for i in candidates if chunks[i]["metadata"]["company"] in (company, "Acme (all)")]
    return candidates[:k]

def rerank(question, candidate_ids, top_n=3):
    """Stage 2: cross-encoder rescoring of (question, chunk) pairs."""
    scores = reranker.predict([(question, chunks[i]["text"]) for i in candidate_ids])
    order = np.argsort(-scores)[:top_n]
    return [(candidate_ids[j], float(scores[j])) for j in order]

question = "How long can I wait before submitting my travel bills?"
cands = retrieve(question)
print("Before rerank (top 5):", [chunks[i]["metadata"]["section"] for i in cands[:5]])
print("After rerank  (top 3):", [(chunks[i]["metadata"]["section"], round(s, 2)) for i, s in rerank(question, cands)])

*(On a small corpus the shortlist is often already right; the reranker shines on large, noisy corpora where the right chunk sits at rank 15.)*

> Our `retrieve()` filters after ranking *everything*, which is equivalent to a pre-filter on a 30-chunk corpus. At scale, use the vector DB's `where=` filter (Section 6.4).

### 8.2 Load the LLM

We use a small open model so the notebook runs anywhere, for free. Prefer an API model? Set `USE_OPENAI = True` and add your key (cell below).

In [ ]:
from transformers import AutoModelForCausalLM

USE_OPENAI = False           # ← set True to use an OpenAI model instead (needs an API key)
OPENAI_MODEL = "gpt-4o-mini" # check current names: https://platform.openai.com/docs/models

if USE_OPENAI:
    !pip install -q openai
    from openai import OpenAI
    from getpass import getpass
    os.environ.setdefault("OPENAI_API_KEY", getpass("OpenAI API key: "))
    oai = OpenAI()
    def generate(messages, max_new_tokens=300):
        r = oai.chat.completions.create(model=OPENAI_MODEL, messages=messages,
                                        max_tokens=max_new_tokens, temperature=0)
        return r.choices[0].message.content.strip()
else:
    gen_tok = AutoTokenizer.from_pretrained(LLM_MODEL)
    gen_model = AutoModelForCausalLM.from_pretrained(LLM_MODEL).to(DEVICE).eval()
    if DEVICE == "cuda":
        gen_model = gen_model.half()

    @torch.no_grad()
    def generate(messages, max_new_tokens=300):
        prompt = gen_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = gen_tok(prompt, return_tensors="pt").to(DEVICE)
        out = gen_model.generate(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"],
                                 max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=gen_tok.eos_token_id)
        return gen_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(generate([{"role": "user", "content": "In one sentence, what is retrieval-augmented generation?"}]))

### 8.3 Augment: build a grounded prompt

The **augmented prompt** = instructions + numbered context chunks + the question. Key instructions:
- answer **only** from the context,
- **cite** the sources,
- say **"I don't know"** when the context doesn't contain the answer (this is how RAG reduces hallucination).

In [ ]:
SYSTEM_PROMPT = (
    "You are Acme Corp's HR and IT assistant. Answer the question using ONLY the numbered context passages. "
    "Cite the passages you used like [1] or [2]. If the context does not contain the answer, reply exactly: "
    "\"I don't know based on the Acme documents.\" Keep the answer short."
)

def build_messages(question, top_chunks):
    context = "\n\n".join(
        f"[{n}] (source: {chunks[i]['metadata']['source']}, section: {chunks[i]['metadata']['section']})\n{chunks[i]['text']}"
        for n, (i, _) in enumerate(top_chunks, start=1))
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}]

msgs = build_messages(question, rerank(question, retrieve(question)))
print(msgs[1]["content"])

### 8.4 Generate: the complete `ask()` function 🎉

In [ ]:
def ask(question, company=None, k_retrieve=20, k_final=3, verbose=True):
    cands = retrieve(question, k=k_retrieve, company=company)    # 1. retrieve (hybrid + filter)
    top = rerank(question, cands, top_n=k_final)                  # 2. rerank (cross-encoder)
    answer = generate(build_messages(question, top))              # 3. augment + generate
    if verbose:
        print("❓", question, f"  (company={company})" if company else "")
        print("💬", answer)
        print("📚 Sources:")
        for n, (i, s) in enumerate(top, start=1):
            m = chunks[i]["metadata"]
            print(f"   [{n}] {m['source']} › {m['section']}  (rerank score {s:.2f})")
        print()
    return answer, top

ask("How much vacation do I get?", company="Acme-India")
ask("How much vacation do I get?", company="Acme-US")
ask("I'm getting error E1047, what should I do?")

In [ ]:
ask("Is my mother covered by the health insurance?")
ask("What is the dress code on Fridays?")              # NOT in our documents → should say "I don't know"
ask("When do my stock options vest?")                  # lives on the scanned page — what happens?

**What to notice:**
- The **same question** gets different, correct answers for India vs US — thanks to metadata.
- The dress-code question should be refused: the model is grounded. *(Small 0.5B models sometimes still improvise — a bigger model follows instructions better. Try `Qwen2.5-1.5B-Instruct` on a GPU.)*
- The stock-option question fails because page 3 was never extracted. **No error anywhere — just a worse answer.** Go back, set `RUN_OCR = True`, re-run from Section 4, and ask again.

🔗 **Read more**
- Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks* (2020): https://arxiv.org/abs/2005.11401 · Meta AI blog: https://ai.meta.com/blog/retrieval-augmented-generation-streamlining-the-creation-of-intelligent-natural-language-processing-models/
- SBERT *Retrieve & Re-Rank* example: https://sbert.net/examples/sentence_transformer/applications/retrieve_rerank/README.html
- Liu et al., *Lost in the Middle* (where in the prompt to put context): https://arxiv.org/abs/2307.03172
- Hugging Face chat templates: https://huggingface.co/docs/transformers/main/en/chat_templating

---
# 9 · ⏩ The same pipeline in LangChain

Everything above in a framework. LangChain gives you standard interfaces (`Document`, `Embeddings`, `VectorStore`, `Retriever`) so you can swap Chroma ↔ FAISS ↔ Qdrant, or MiniLM ↔ OpenAI embeddings, by changing one line.

| Our hand-written code | LangChain equivalent |
|---|---|
| `chunks` list of dicts | `langchain_core.documents.Document(page_content, metadata)` |
| `SentenceTransformer(...).encode` | `langchain_huggingface.HuggingFaceEmbeddings` |
| `chromadb` collection | `langchain_chroma.Chroma` |
| `chroma_search(..., where=)` | `vectorstore.as_retriever(search_kwargs={"k": 3, "filter": {...}})` |
| `build_messages` + `generate` | `ChatPromptTemplate` → LLM → `StrOutputParser` (an **LCEL** chain) |

In [ ]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

lc_docs = [Document(page_content=c["text"], metadata=c["metadata"], id=c["id"]) for c in chunks]

lc_embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL,
                                      model_kwargs={"device": DEVICE},
                                      encode_kwargs={"normalize_embeddings": True})

vectorstore = Chroma.from_documents(lc_docs, lc_embeddings, collection_name="acme_langchain",
                                    collection_metadata={"hnsw:space": "cosine"})

for doc, dist in vectorstore.similarity_search_with_score("When does the canteen open?", k=2):
    print(f"{1 - dist:.3f}", doc.metadata["section"], "|", doc.page_content[:70])

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

retriever = vectorstore.as_retriever(search_kwargs={"k": 3, "filter": {"company": {"$in": ["Acme-India", "Acme (all)"]}}})

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

def format_docs(docs):
    return "\n\n".join(f"[{n}] (source: {d.metadata['source']}) {d.page_content}" for n, d in enumerate(docs, 1))

def call_llm(prompt_value):
    """Adapter: LangChain prompt → our generate() (local Qwen or OpenAI)."""
    roles = {"system": "system", "human": "user", "ai": "assistant"}
    return generate([{"role": roles[m.type], "content": m.content} for m in prompt_value.to_messages()])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | RunnableLambda(call_llm)
    | StrOutputParser()
)

print(rag_chain.invoke("How many sick days do I get?"))

> **Using a hosted chat model instead?** Replace `RunnableLambda(call_llm)` with e.g. `ChatOpenAI(model="gpt-4o-mini")` from `langchain-openai`, or use `init_chat_model("provider:model")` from `langchain` — the rest of the chain is unchanged.
>
> **Want FAISS instead of Chroma?** `from langchain_community.vectorstores import FAISS; FAISS.from_documents(lc_docs, lc_embeddings)` — note: FAISS in LangChain supports simple metadata filters only.

🔗 **Read more — LangChain**
- RAG tutorial: https://docs.langchain.com/oss/python/langchain/rag · Semantic search / knowledge-base tutorial: https://docs.langchain.com/oss/python/langchain/knowledge-base
- Integrations: text splitters https://docs.langchain.com/oss/python/integrations/splitters · embeddings https://docs.langchain.com/oss/python/integrations/embeddings · vector stores https://docs.langchain.com/oss/python/integrations/vectorstores · retrievers https://docs.langchain.com/oss/python/integrations/retrievers · chat models https://docs.langchain.com/oss/python/integrations/chat
- API reference: https://reference.langchain.com/python/
- Evaluating a RAG app with LangSmith: https://docs.langchain.com/langsmith/evaluate-rag-tutorial
- Alternative framework — LlamaIndex: https://docs.llamaindex.ai · Haystack: https://docs.haystack.deepset.ai

---
# 10 · ⏩ Mini bake-off: which retriever is best **on our data**?

The lecture's rule: *don't pick by the leaderboard — measure on your own queries.* Here's the smallest possible evaluation: 12 questions, each with a keyword that must appear in a retrieved chunk. We measure **hit-rate@3** (did a correct chunk land in the top 3?).

In [ ]:
EVAL = [  # (question, keyword that must appear in a correct chunk)
    ("How much vacation do I get?",                         "paid leave"),
    ("error E1047 when I log in",                           "E1047"),
    ("Can I work from home?",                               "work from home"),
    ("When does the canteen open?",                         "8 am"),
    ("How long do I have to file my travel reimbursement?", "within 30 days"),
    ("What's the storage limit on my home drive?",          "50 GB"),
    ("How many weeks of maternity leave are there?",        "26 weeks"),
    ("Is my family covered by medical insurance?",          "dependants"),
    ("How often must I change my password?",                "90 days"),
    ("Which VPN gateway should I use in Pune?",             "India-West"),
    ("What happens to leave I don't use?",                  "carr"),        # carries / carry
    ("When do my stock options vest?",                      "vest"),        # only findable after OCR!
]

def hit_rate(rank_fn, k=3):
    hits = [any(kw.lower() in chunks[i]["text"].lower() for i in rank_fn(q)[:k]) for q, kw in EVAL]
    return np.mean(hits), hits

def hybrid_rerank(q):
    return [i for i, _ in rerank(q, retrieve(q, k=20), top_n=3)]

results = {}
for name, fn in [("BM25", bm25_ranking), ("Dense", dense_ranking),
                 ("Hybrid (RRF)", hybrid_ranking), ("Hybrid + rerank", hybrid_rerank)]:
    results[name] = hit_rate(fn)
    print(f"{name:<16} hit-rate@3 = {results[name][0]:.2f}")

print("\nPer question (✓ = found)   columns:", " | ".join(results))
for (q, _), *row in zip(EVAL, *[r[1] for r in results.values()]):
    print("  " + " ".join("✓" if h else "·" for h in row), " ", q)

**What to notice:** each method fails on *different* questions — that's why hybrid wins. And the stock-option question fails for **every** retriever: no model swap can fix a document-processing bug. *Fix the pipeline before you swap the model.*

🔗 **Read more — evaluation**
- RAGAS (faithfulness, context precision/recall): https://docs.ragas.io
- BEIR benchmark for retrieval: https://github.com/beir-cellar/beir
- MTEB: https://github.com/embeddings-benchmark/mteb

---
# 11 · The three silent bugs (checklist)

| # | Bug | Symptom | Where you saw it here | Fix |
|---|---|---|---|---|
| 1 | **Empty chunks in prod** | scanned pages return `""` from `pypdf` | Section 4 (page 3), Section 10 (stock options) | check the text layer **page by page**; OCR the empty ones |
| 2 | **Missing `query:` / `passage:` prefixes** | recall quietly drops | Section 3.5 (E5) | read the model card; add the exact prefixes |
| 3 | **Wrong distance metric** | quietly worse ranking | Section 6.2 | normalise + inner product (= cosine), or set `hnsw:space="cosine"` |

None of these raise an error. They just make answers worse. **Check your configuration before you reach for a bigger model.**

## ✅ Summary
1. **Sparse (BM25)** matches strings; **dense** matches meaning; **hybrid + RRF** gets both.
2. **Bi-encoders** retrieve (fast, pre-computed vectors); **cross-encoders** rerank (slow, accurate).
3. **Document processing** and **chunking** cap everything downstream.
4. **FAISS** is a fast search *library*; **Chroma** is a *database* (text + metadata + filters). Pre-filter, don't post-filter.
5. Measure on **your own questions** — even 12 of them tell you a lot.

---
# 🏋️ Exercises (try at home)

1. **OCR fix.** Set `RUN_OCR = True`, re-run from Section 4, and re-run the bake-off. Does the stock-option question now pass?
2. **Chunk size.** Replace the Markdown-header chunks with `sliding_window(..., size=32, overlap=0)` and with `size=256`. How does hit-rate@3 change? Why?
3. **Swap the embedder.** Try `BAAI/bge-small-en-v1.5` (needs the query instruction from its model card!) or `intfloat/e5-small-v2` with `query:`/`passage:` prefixes. Re-run the bake-off. Which wins on *our* data?
4. **Multilingual.** Ask *"मुझे कितनी छुट्टियां मिलती हैं?"* (Hindi: how many leave days do I get?). Does MiniLM find the leave policy? Try `BAAI/bge-m3` or `intfloat/multilingual-e5-small`.
5. **Remove the contextual headers** (`f"{title} › {section}: "`) from the chunk text. What happens to the India-vs-US question without a filter?
6. **Persist the index.** Switch to `chromadb.PersistentClient(path="acme_db")`, restart the runtime, and load the collection without re-embedding.
7. **Bigger corpus.** Index a real PDF of your choice (e.g. a research paper) with `extract_pages()` + the recursive splitter. Ask it five questions.
8. **(Advanced) Text-to-SQL.** Put a small table of employees in SQLite and let the LLM write SQL for *"How many employees joined in 2024?"* instead of using embeddings.

---
## 📚 All links in one place

**Papers**
- RAG — Lewis et al. 2020: https://arxiv.org/abs/2005.11401
- Sentence-BERT — Reimers & Gurevych 2019: https://arxiv.org/abs/1908.10084
- Dense Passage Retrieval — Karpukhin et al. 2020: https://arxiv.org/abs/2004.04906
- E5 — Wang et al. 2022: https://arxiv.org/abs/2212.03533
- Matryoshka Representation Learning — Kusupati et al. 2022: https://arxiv.org/abs/2205.13147
- MiniLMv2 — Wang et al. 2020: https://arxiv.org/abs/2012.15828
- Product Quantization — Jégou et al. 2011: https://ieeexplore.ieee.org/document/5432202
- HNSW — Malkov & Yashunin 2016: https://arxiv.org/abs/1603.09320
- The Faiss library — Douze et al. 2024: https://arxiv.org/abs/2401.08281
- Reciprocal Rank Fusion — Cormack et al. 2009: https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf
- BM25 and beyond — Robertson & Zaragoza 2009: https://www.nowpublishers.com/article/Details/INR-019
- Lost in the Middle — Liu et al. 2023: https://arxiv.org/abs/2307.03172

**Libraries & docs**
- Sentence-Transformers: https://sbert.net · Hugging Face Transformers: https://huggingface.co/docs/transformers · Hub: https://huggingface.co/models
- FAISS: https://github.com/facebookresearch/faiss/wiki · ChromaDB: https://docs.trychroma.com · rank_bm25: https://github.com/dorianbrown/rank_bm25
- LangChain: https://docs.langchain.com/oss/python/langchain/overview · LlamaIndex: https://docs.llamaindex.ai · Haystack: https://docs.haystack.deepset.ai
- pypdf: https://pypdf.readthedocs.io · pdfplumber: https://github.com/jsvine/pdfplumber · docling: https://github.com/docling-project/docling · Unstructured: https://docs.unstructured.io · trafilatura: https://trafilatura.readthedocs.io
- RAGAS (evaluation): https://docs.ragas.io · MTEB leaderboard: https://huggingface.co/spaces/mteb/leaderboard

**Tutorials & visual explainers**
- Pinecone — FAISS course: https://www.pinecone.io/learn/series/faiss/ · Chunking strategies: https://www.pinecone.io/learn/chunking-strategies/
- RAG notes with diagrams (used in the lecture): https://github.com/hkproj/retrieval-augmented-generation-notes
- Anthropic — Contextual Retrieval: https://www.anthropic.com/news/contextual-retrieval
- Hugging Face — Matryoshka embeddings: https://huggingface.co/blog/matryoshka